# Mu-SHROOM (English): XLM-RoBERTa-Large token classification

Fine-tune **`xlm-roberta-large`** on **`Helsinki-NLP/mu-shroom`** config **`en`** to label **hallucinated vs non-hallucinated** spans using **`hard_labels`** (character spans → BIO tags).

**Before running:** Runtime → **Change runtime type** → **GPU** (T4 is enough).

Outputs are saved under **`/content/`** on the Colab VM (see the cell *Where are my saved files?* after training).

**Fixes baked in:** class‑weighted loss (stops predicting only `O`), token‑level F1 for B/I‑HALL + `f1_hall_mean` for checkpoint selection, seqeval `zero_division=0`, more epochs on tiny data, GPU device fix in inference, `scikit-learn` for metrics. Optional: set `HF_TOKEN` in Colab secrets for faster Hub downloads (not required for public assets).

In [1]:
# Install dependencies (Colab) — pin stack so save/load LayerNorm names match train vs inference
!pip install -q "datasets>=2.18.0" "transformers>=4.41.0" "accelerate>=0.29.0" evaluate seqeval scikit-learn

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
torch: 2.10.0+cu128
cuda available: True
GPU: NVIDIA A100-SXM4-80GB


In [2]:
import inspect
import os

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

OUTPUT_DIR = "/content/xlmr-mu-shroom-en"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = "xlm-roberta-large"
MAX_LENGTH = 512  # Increased to handle longer texts
SEED = 1234       # New seed for different random initialization
NUM_EPOCHS = 20   # Increased epochs (best checkpoint is saved anyway)

label_list = ["O", "B-HALL", "I-HALL"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

seqeval = evaluate.load("seqeval")


def spans_to_token_bio(offsets, spans):
    spans = spans or []
    spans = sorted(spans, key=lambda x: (x[0], x[1]))

    def token_overlaps_span(tok_s, tok_e, sp_s, sp_e):
        return (tok_s < sp_e) and (tok_e > sp_s)

    labels = []
    for tok_s, tok_e in offsets:
        if tok_s == tok_e == 0:
            labels.append(-100)
            continue
        matched = None
        for sp_s, sp_e in spans:
            if token_overlaps_span(tok_s, tok_e, sp_s, sp_e):
                matched = (sp_s, sp_e)
                break
        if matched is None:
            labels.append(label2id["O"])
            continue
        sp_s, sp_e = matched
        begins = tok_s <= sp_s < tok_e or tok_s == sp_s
        labels.append(label2id["B-HALL"] if begins else label2id["I-HALL"])

    prev_inside = False
    for i, lab in enumerate(labels):
        if lab in (-100, label2id["O"]):
            prev_inside = False
            continue
        if lab == label2id["I-HALL"] and not prev_inside:
            labels[i] = label2id["B-HALL"]
        prev_inside = True
    return labels


def load_english_mu_shroom():
    try:
        return load_dataset("Helsinki-NLP/mu-shroom", "en")
    except Exception:
        ds = load_dataset("Helsinki-NLP/mu-shroom", "all")
        for c in ["language", "lang", "locale"]:
            if "validation" in ds and c in ds["validation"].column_names:
                return ds.filter(lambda x: x[c] == "en")
        raise RuntimeError("Could not filter English from 'all' config.")


def make_training_args(output_dir, train_bs, eval_bs, epochs, lr, wd, seed, use_fp16):
    """Best checkpoint tracks token-level F1 on minority (Hall) classes, not plain accuracy."""
    sig = inspect.signature(TrainingArguments.__init__)
    p = sig.parameters
    kw = dict(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        weight_decay=wd,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="f1_hall_mean",
        greater_is_better=True,
        fp16=use_fp16,
        seed=seed,
        report_to="none",
        save_total_limit=2,  # Added to prevent disk space issues
    )
    if "evaluation_strategy" in p:
        kw["evaluation_strategy"] = "epoch"
        kw["save_strategy"] = "epoch"
    else:
        if "eval_strategy" in p:
            kw["eval_strategy"] = "epoch"
        if "save_strategy" in p:
            kw["save_strategy"] = "epoch"
    return TrainingArguments(**kw)


def collect_flat_labels(dataset, label_key="labels"):
    y = []
    for row in dataset[label_key]:
        y.extend(int(t) for t in row if t != -100)
    return np.array(y, dtype=np.int64)


def make_class_weights(train_ds, num_labels):
    y = collect_flat_labels(train_ds)
    classes = np.arange(num_labels)
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return torch.tensor(w, dtype=torch.float32)


class WeightedTrainer(Trainer):
    """Cross-entropy with class weights (fixes all-O collapse from imbalance).
    Trainer is not an nn.Module, so we cannot use register_buffer — store weights as attributes.
    """

    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce_weight = class_weights.detach().clone()

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self._ce_weight.to(device=logits.device, dtype=logits.dtype)
        loss_fct = nn.CrossEntropyLoss(weight=w, ignore_index=-100)
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
ds = load_english_mu_shroom()
print(ds)

tmp = ds["validation"].train_test_split(test_size=0.2, seed=SEED) # 20% test size to boost training data slightly
train_split = tmp["train"]
eval_split = tmp["test"]  # held-out validation (no overlap with train)

text_col, spans_col = "model_output_text", "hard_labels"


def preprocess(batch):
    enc = tokenizer(
        batch[text_col],
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )
    enc["labels"] = [
        spans_to_token_bio(off, sp) for off, sp in zip(enc["offset_mapping"], batch[spans_col])
    ]
    enc.pop("offset_mapping")
    return enc


train_tok = train_split.map(preprocess, batched=True, remove_columns=train_split.column_names)
eval_tok = eval_split.map(preprocess, batched=True, remove_columns=eval_split.column_names)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer)


def compute_metrics(p):
    """Seqeval (entity) + token F1 for B-HALL and I-HALL + mean 'Hall' F1 for checkpoint selection."""
    preds, labels = p
    preds = np.argmax(preds, axis=-1)

    true_predictions, true_labels = [], []
    flat_pred, flat_true = [], []
    for pred_row, lab_row in zip(preds, labels):
        cp, cl = [], []
        for pr, lb in zip(pred_row, lab_row):
            if lb == -100:
                continue
            cp.append(id2label[int(pr)])
            cl.append(id2label[int(lb)])
            flat_pred.append(int(pr))
            flat_true.append(int(lb))
        true_predictions.append(cp)
        true_labels.append(cl)

    flat_pred = np.array(flat_pred)
    flat_true = np.array(flat_true)

    # Token-level F1 (handles imbalance better than accuracy)
    f1_macro = float(f1_score(flat_true, flat_pred, average="macro", zero_division=0))
    f1_w = float(f1_score(flat_true, flat_pred, average="weighted", zero_division=0))
    f1_o = float(f1_score(flat_true, flat_pred, labels=[0], average="macro", zero_division=0))
    f1_b = float(f1_score(flat_true, flat_pred, labels=[1], average="macro", zero_division=0))
    f1_i = float(f1_score(flat_true, flat_pred, labels=[2], average="macro", zero_division=0))
    f1_hall_mean = (f1_b + f1_i) / 2.0

    out = {
        "f1_macro": f1_macro,
        "f1_weighted": f1_w,
        "f1_O": f1_o,
        "f1_B-HALL": f1_b,
        "f1_I-HALL": f1_i,
        "f1_hall_mean": f1_hall_mean,
    }

    try:
        se = seqeval.compute(
            predictions=true_predictions,
            references=true_labels,
            zero_division=0,
        )
    except TypeError:
        se = seqeval.compute(
            predictions=true_predictions,
            references=true_labels,
        )
    out.update(se)
    return out


class_weights = make_class_weights(train_tok, len(label_list))
print("Class weights (O, B-HALL, I-HALL):", class_weights.tolist())

use_fp16 = torch.cuda.is_available()
args = make_training_args(
    output_dir=OUTPUT_DIR,
    train_bs=4,
    eval_bs=4,
    epochs=NUM_EPOCHS,
    lr=2e-5,
    wd=0.01,
    seed=SEED,
    use_fp16=use_fp16,
)

trainer_kwargs = dict(
    class_weights=class_weights,
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
try:
    trainer = WeightedTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = WeightedTrainer(tokenizer=tokenizer, **trainer_kwargs)

trainer.train()

best_dir = os.path.join(OUTPUT_DIR, "best")
os.makedirs(best_dir, exist_ok=True)
trainer.model.save_pretrained(best_dir, safe_serialization=True)
tokenizer.save_pretrained(best_dir)
print("Saved to:", best_dir)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/xlmr-mu-shroom-en/best


## Where are my saved files?

Training writes to the **Colab VM’s local disk**, not to your Google Drive unless you copy them.

- **Path on the VM:** `/content/xlmr-mu-shroom-en/best/` (weights, tokenizer, `config.json`).
- **See them in the UI:** click the **folder icon** in the left sidebar → open **`content`** → **`xlmr-mu-shroom-en`** → **`best`**.
- If the folder is empty, expand **`xlmr-mu-shroom-en`** (not only `best`): checkpoints also live under `OUTPUT_DIR` next to `best`.
- **Important:** `/content/` is **deleted when the runtime disconnects**. Download or copy to Drive before closing the session.

Run the next cell to list files and sizes. Use it to confirm the save worked.

In [3]:
import os

out = "/content/xlmr-mu-shroom-en"
best = os.path.join(out, "best")

def hum(n):
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024.0 or u == "TB":
            return f"{n:.2f} {u}"
        n /= 1024.0

print("Exists:", os.path.isdir(best), "→", best)
if os.path.isdir(out):
    print("\nTop-level under", out, ":")
    for name in sorted(os.listdir(out)):
        p = os.path.join(out, name)
        print(" ", name, "/" if os.path.isdir(p) else "", flush=True)
if os.path.isdir(best):
    print("\nFiles in best/:")
    tot = 0
    for fn in sorted(os.listdir(best)):
        p = os.path.join(best, fn)
        if os.path.isfile(p):
            s = os.path.getsize(p)
            tot += s
            print(f"  {fn:40s}  {hum(s)}")
        else:
            print(f"  {fn}/  (dir)")
    print(f"\nSum of files in best/: ~{hum(tot)}")
else:
    print("best/ not found — training save may have failed or path differs.")

# Optional: download the whole folder as a zip (large ~2GB+; may take minutes)
# SKIP = False
# if not SKIP and os.path.isdir(best):
#     import shutil
#     from google.colab import files
#     zip_path = shutil.make_archive("/content/xlmr-mu-shroom-en-best", "zip", best)
#     files.download(zip_path)

Exists: True → /content/xlmr-mu-shroom-en/best

Top-level under /content/xlmr-mu-shroom-en :
  best /
  checkpoint-100 /
  checkpoint-200 /

Files in best/:
  config.json                               898.00 B
  model.safetensors                         2.08 GB
  tokenizer.json                            16.00 MB
  tokenizer_config.json                     314.00 B

Sum of files in best/: ~2.10 GB


## Inference: spans on new text

In [4]:
import os
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

OUTPUT_DIR = "/content/xlmr-mu-shroom-en"
best_dir = os.path.join(OUTPUT_DIR, "best")
tok = AutoTokenizer.from_pretrained(best_dir, use_fast=True)
mdl = AutoModelForTokenClassification.from_pretrained(
    best_dir,
    ignore_mismatched_sizes=True,
)
mdl.eval()
device = next(mdl.parameters()).device
id2l = mdl.config.id2label

text = "Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China."
enc = tok(text, return_tensors="pt", return_offsets_mapping=True, truncation=True, max_length=256)
offsets = enc.pop("offset_mapping")[0].tolist()
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    logits = mdl(**enc).logits[0]
pred_ids = logits.argmax(-1).tolist()

# Quick sanity: count predicted labels on this sentence
from collections import Counter
cnt = Counter(id2l[int(i)] for i in pred_ids)
print("Predicted label counts:", dict(cnt))

spans = []
cur_s = cur_e = None
for (s, e), pid in zip(offsets, pred_ids):
    if s == e == 0:
        continue
    lab = id2l[int(pid)]
    if lab == "B-HALL":
        if cur_s is not None:
            spans.append([cur_s, cur_e])
        cur_s, cur_e = s, e
    elif lab == "I-HALL" and cur_s is not None:
        cur_e = e
    else:
        if cur_s is not None:
            spans.append([cur_s, cur_e])
            cur_s = cur_e = None
if cur_s is not None:
    spans.append([cur_s, cur_e])

print("Predicted hallucination spans [start, end):", spans)
for a, b in spans:
    print(repr(text[a:b]))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Predicted label counts: {'O': 15, 'B-HALL': 4, 'I-HALL': 2}
Predicted hallucination spans [start, end): [[25, 31], [45, 49], [69, 83]]
'silver'
'2008'
'Beijing, China'


## Official scoring

This section evaluates the fine-tuned model with the official Mu-SHROOM scorer and prints a consolidated results table in the same format used by the baseline notebook (`Model`, `IoU`, `Correlation`, `Prediction file`).

In [5]:
# Official scoring setup (participant kit + scorer helpers)
import os
import sys
import json
import subprocess
import importlib.util
from typing import Optional, Tuple

import pandas as pd
from datasets import load_dataset


def json_default(obj):
    if hasattr(obj, "item"):
        return obj.item()
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Type not serializable: {type(obj)}")


def clone_if_missing(url: str, path: str):
    if os.path.isdir(path):
        print(f"Using existing {path}/")
        return
    subprocess.check_call(["git", "clone", "--depth", "1", url, path])


clone_if_missing("https://github.com/Helsinki-NLP/mu-shroom", "mu-shroom")
KIT = os.path.abspath("mu-shroom/participant_kit")
assert os.path.isfile(os.path.join(KIT, "scorer.py")), f"Expected scorer at {KIT}/scorer.py"
print("Participant kit:", KIT)


def load_scorer_module():
    path = os.path.join(KIT, "scorer.py")
    spec = importlib.util.spec_from_file_location("mu_shroom_scorer", path)
    mod = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod


scorer = load_scorer_module()


def write_jsonl(path: str, rows):
    with open(path, "w", encoding="utf-8") as w:
        for r in rows:
            w.write(json.dumps(r, ensure_ascii=False, default=json_default) + "\n")


def score_predictions(ref_jsonl: str, pred_jsonl: str, scores_txt: Optional[str] = "scores_last_run.txt") -> Tuple[float, float]:
    """Returns mean IoU and mean Spearman correlation from official scorer.py."""
    ref_list = scorer.load_jsonl_file_to_records(ref_jsonl, is_ref=True)
    pred_list = scorer.load_jsonl_file_to_records(pred_jsonl, is_ref=False)
    ious, cors = scorer.main(ref_list, pred_list, scores_txt)
    return float(ious.mean()), float(cors.mean())


# Build official reference file from Mu-SHROOM English validation split.
ref_ds = load_dataset("Helsinki-NLP/mu-shroom", "en")["validation"]
REF_JSONL = os.path.abspath("reference_en_validation.jsonl")
write_jsonl(REF_JSONL, [
    {
        "id": row["id"],
        "model_output_text": row["model_output_text"],
        "soft_labels": row["soft_labels"],
        "hard_labels": row["hard_labels"],
    }
    for row in ref_ds
])
print("Reference rows:", len(ref_ds))
print("Reference file:", REF_JSONL)

Participant kit: /content/mu-shroom/participant_kit
Reference rows: 50
Reference file: /content/reference_en_validation.jsonl


In [6]:
# Official scoring run + consolidated table (baseline-format columns)
import torch
import torch.nn.functional as F
from transformers import AutoModelForTokenClassification, AutoTokenizer


def fine_tuned_predict_jsonl(checkpoint_dir: str, ref_jsonl: str, out_jsonl: str, tokenizer_name: Optional[str] = None):
    """Token-level predictions -> character spans + soft labels for official scorer."""
    tok_name = tokenizer_name or checkpoint_dir
    tokenizer = AutoTokenizer.from_pretrained(tok_name, use_fast=True)
    model = AutoModelForTokenClassification.from_pretrained(checkpoint_dir)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    )
    model.to(device)
    model.eval()

    records = []
    with open(ref_jsonl, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    out_rows = []
    with torch.no_grad():
        for rec in records:
            text = rec["model_output_text"]
            enc = tokenizer(
                text,
                return_offsets_mapping=True,
                return_tensors="pt",
                truncation=True,
                max_length=512,
            )
            offset_mapping = enc.pop("offset_mapping")[0]
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits[0]
            probs = F.softmax(logits, dim=-1)
            preds = torch.argmax(logits, dim=-1)

            soft_labels = []
            hard_labels = []
            for j, offset in enumerate(offset_mapping):
                s, e = offset[0].item(), offset[1].item()
                if s >= e:
                    continue

                # Prefer hall probability if available; fallback to non-O mass for safety.
                if probs.shape[-1] > 2:
                    p_hall = (probs[j, 1] + probs[j, 2]).item()
                    pred_is_hall = preds[j].item() in (1, 2)
                else:
                    p_hall = probs[j, 1].item() if probs.shape[-1] > 1 else 0.0
                    pred_is_hall = preds[j].item() == 1

                soft_labels.append({"start": s, "end": e, "prob": p_hall})
                if pred_is_hall:
                    hard_labels.append([s, e])

            out_rows.append({"id": rec["id"], "soft_labels": soft_labels, "hard_labels": hard_labels})

    write_jsonl(out_jsonl, out_rows)
    print("Wrote", len(out_rows), "predictions to", out_jsonl)


# Resolve best fine-tuned checkpoint location saved earlier in this notebook.
OUTPUT_DIR = "/content/xlmr-mu-shroom-en"
best_dir = os.path.join(OUTPUT_DIR, "best")
if not os.path.isdir(best_dir):
    raise FileNotFoundError(f"Expected fine-tuned model dir at {best_dir}.")

pred_ft = "predictions_finetuned_official.jsonl"
iou_ft, cor_ft = float("nan"), float("nan")
try:
    fine_tuned_predict_jsonl(best_dir, REF_JSONL, pred_ft)
    iou_ft, cor_ft = score_predictions(REF_JSONL, pred_ft, "scores_finetuned_official.txt")
    print(f"Fine-tuned (official) -> IoU: {iou_ft:.8f}  Cor: {cor_ft:.8f}")
except Exception as e:
    print("Fine-tuned scoring failed:", repr(e))


# Baseline-format consolidated table.
rows = []
if os.path.isfile(pred_ft):
    rows.append({
        "Model": "Fine-tuned XLM-R",
        "IoU": iou_ft,
        "Correlation": cor_ft,
        "Prediction file": pred_ft,
    })

# If baseline prediction files are present in the same runtime, include them too.
for name, pred_file, score_file in [
    ("XLM-R", "predictions_xlmr.jsonl", "scores_xlmr.txt"),
    ("Random", "predictions_random.jsonl", "scores_random.txt"),
    ("Mark-none (sanity)", "predictions_mark_none.jsonl", "scores_mark_none.txt"),
    ("Mark-all (sanity)", "predictions_mark_all.jsonl", "scores_mark_all.txt"),
]:
    if os.path.isfile(pred_file):
        iou, cor = score_predictions(REF_JSONL, pred_file, score_file)
        rows.append({"Model": name, "IoU": iou, "Correlation": cor, "Prediction file": pred_file})

if rows:
    df = pd.DataFrame(rows).sort_values(by="IoU", ascending=False).reset_index(drop=True)
    display(df)
else:
    print("No prediction files found yet.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Wrote 50 predictions to predictions_finetuned_official.jsonl
Fine-tuned (official) -> IoU: 0.72292907  Cor: 0.51758484


,Model,IoU,Correlation,Prediction file
0,Fine-tuned XLM-R,0.722929,0.517585,predictions_finetuned_official.jsonl


## Official scoring optimization (English)

This section tunes inference/post-processing for the **official scorer** on English validation data:
- temperature scaling for soft-label probabilities,
- thresholding for hallucination spans,
- small span cleanup + gap merge.

It keeps training unchanged and improves the prediction format/objective alignment for IoU and correlation.

In [7]:
# English-only official metric optimization (IoU + Correlation)
import itertools
import math
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForTokenClassification, AutoTokenizer


# ---------- helpers ----------
def _merge_spans(spans: List[List[int]], gap: int = 0) -> List[List[int]]:
    """Merge overlapping/nearby [start,end] char spans."""
    if not spans:
        return []
    spans = sorted([[int(s), int(e)] for s, e in spans if e > s], key=lambda x: (x[0], x[1]))
    out = [spans[0]]
    for s, e in spans[1:]:
        ps, pe = out[-1]
        if s <= pe + gap:
            out[-1][1] = max(pe, e)
        else:
            out.append([s, e])
    return out


def _filtered_spans(spans: List[List[int]], min_len: int, merge_gap: int) -> List[List[int]]:
    spans = [sp for sp in spans if (sp[1] - sp[0]) >= min_len]
    return _merge_spans(spans, gap=merge_gap)


def _softmax_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    t = max(float(temperature), 1e-6)
    return F.softmax(logits / t, dim=-1)


def predict_with_params(
    checkpoint_dir: str,
    ref_jsonl: str,
    out_jsonl: str,
    threshold: float,
    temperature: float,
    min_span_len: int,
    merge_gap: int,
    max_length: int = 512,
):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir, use_fast=True)
    model = AutoModelForTokenClassification.from_pretrained(checkpoint_dir)
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    )
    model.to(device)
    model.eval()

    records = []
    with open(ref_jsonl, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    out_rows = []
    with torch.no_grad():
        for rec in records:
            text = rec["model_output_text"]
            enc = tokenizer(
                text,
                return_offsets_mapping=True,
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
            )
            offset_mapping = enc.pop("offset_mapping")[0]
            enc = {k: v.to(device) for k, v in enc.items()}

            logits = model(**enc).logits[0]
            probs = _softmax_temperature(logits, temperature)

            hard_spans = []
            soft_labels = []
            for j, offset in enumerate(offset_mapping):
                s, e = offset[0].item(), offset[1].item()
                if s >= e:
                    continue

                if probs.shape[-1] > 2:
                    p_hall = float((probs[j, 1] + probs[j, 2]).item())
                elif probs.shape[-1] == 2:
                    p_hall = float(probs[j, 1].item())
                else:
                    p_hall = 0.0

                soft_labels.append({"start": s, "end": e, "prob": p_hall})
                if p_hall >= threshold:
                    hard_spans.append([s, e])

            hard_spans = _filtered_spans(hard_spans, min_len=min_span_len, merge_gap=merge_gap)
            out_rows.append({"id": rec["id"], "soft_labels": soft_labels, "hard_labels": hard_spans})

    write_jsonl(out_jsonl, out_rows)


# ---------- fast path (fixed best params from latest sweep) ----------
OUTPUT_DIR = "/content/xlmr-mu-shroom-en"
best_dir = os.path.join(OUTPUT_DIR, "best")
if not os.path.isdir(best_dir):
    raise FileNotFoundError(f"Expected fine-tuned model dir at {best_dir}.")

# Best params found in this notebook's latest search:
# (thr=0.6, T=0.8, min_len=1, gap=2)
BEST_THRESHOLD = 0.60
BEST_TEMPERATURE = 0.80
BEST_MIN_SPAN_LEN = 1
BEST_MERGE_GAP = 2

best_official_pred = "predictions_finetuned_official_optimized.jsonl"
predict_with_params(
    checkpoint_dir=best_dir,
    ref_jsonl=REF_JSONL,
    out_jsonl=best_official_pred,
    threshold=BEST_THRESHOLD,
    temperature=BEST_TEMPERATURE,
    min_span_len=BEST_MIN_SPAN_LEN,
    merge_gap=BEST_MERGE_GAP,
)

best_iou, best_cor = score_predictions(REF_JSONL, best_official_pred, "scores_finetuned_official_optimized.txt")
print(
    f"Fine-tuned optimized (fixed params) -> IoU: {best_iou:.8f}  Cor: {best_cor:.8f}  "
    f"(thr={BEST_THRESHOLD}, T={BEST_TEMPERATURE}, min_len={BEST_MIN_SPAN_LEN}, gap={BEST_MERGE_GAP})"
)

# Final baseline-format comparison table.
final_rows = [
    {
        "Model": "Fine-tuned XLM-R (optimized)",
        "IoU": best_iou,
        "Correlation": best_cor,
        "Prediction file": best_official_pred,
    }
]

for name, pred_file, score_file in [
    ("Fine-tuned XLM-R", "predictions_finetuned_official.jsonl", "scores_finetuned_official.txt"),
    ("XLM-R", "predictions_xlmr.jsonl", "scores_xlmr.txt"),
    ("Random", "predictions_random.jsonl", "scores_random.txt"),
    ("Mark-none (sanity)", "predictions_mark_none.jsonl", "scores_mark_none.txt"),
    ("Mark-all (sanity)", "predictions_mark_all.jsonl", "scores_mark_all.txt"),
]:
    if os.path.isfile(pred_file):
        iou, cor = score_predictions(REF_JSONL, pred_file, score_file)
        final_rows.append({"Model": name, "IoU": iou, "Correlation": cor, "Prediction file": pred_file})

final_df = pd.DataFrame(final_rows).sort_values(by="IoU", ascending=False).reset_index(drop=True)
print("Final official comparison (baseline-format):")
display(final_df)



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Fine-tuned optimized (fixed params) -> IoU: 0.77018292  Cor: 0.51660888  (thr=0.6, T=0.8, min_len=1, gap=2)
Final official comparison (baseline-format):


,Model,IoU,Correlation,Prediction file
0,Fine-tuned XLM-R (optimized),0.770183,0.516609,predictions_finetuned_official_optimized.jsonl
1,Fine-tuned XLM-R,0.722929,0.517585,predictions_finetuned_official.jsonl


# Cross-lingual Mu-SHROOM evaluation (`hi`)

This notebook is now **English-transfer only** for a fast proposal-aligned run on `hi` (Hindi).

It does:
1. Evaluate the English fine-tuned model on the target language.
2. Score predictions with the official Mu-SHROOM scorer.
3. Output per-language IoU/Correlation tables for direct comparison.

In [8]:
# Install dependencies (Colab-safe)
!pip install -q "datasets>=2.18.0" "transformers>=4.41.0" "accelerate>=0.29.0" evaluate seqeval scikit-learn

import os
import json
import subprocess
import importlib.util
from typing import List, Tuple

import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
cuda available: True
GPU: NVIDIA A100-SXM4-80GB


In [9]:
import os

# Config (EN-transfer only)
MODEL_NAME = "xlm-roberta-large"
TARGET_LANGS = ["hi"]

# English model path (from your existing Fine_Tuning notebook run)
EN_MODEL_DIR = "/content/xlmr-mu-shroom-en/best"
MAX_LENGTH = 512

# Optimized official-scoring params from EN tuning
BEST_THRESHOLD = 0.60
BEST_TEMPERATURE = 0.80
BEST_MIN_SPAN_LEN = 1
BEST_MERGE_GAP = 2

OUT_ROOT = "/content/mushroom_multilingual"
os.makedirs(OUT_ROOT, exist_ok=True)


In [10]:
# Official scorer setup + shared helpers

def clone_if_missing(url: str, path: str):
    if not os.path.isdir(path):
        subprocess.check_call(["git", "clone", "--depth", "1", url, path])


def load_scorer_module(kit_dir: str):
    path = os.path.join(kit_dir, "scorer.py")
    spec = importlib.util.spec_from_file_location("mu_shroom_scorer", path)
    mod = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod


def write_jsonl(path: str, rows):
    with open(path, "w", encoding="utf-8") as w:
        for r in rows:
            w.write(json.dumps(r, ensure_ascii=False) + "\n")


def score_predictions(scorer_mod, ref_jsonl: str, pred_jsonl: str, scores_txt: str) -> Tuple[float, float]:
    ref_list = scorer_mod.load_jsonl_file_to_records(ref_jsonl, is_ref=True)
    pred_list = scorer_mod.load_jsonl_file_to_records(pred_jsonl, is_ref=False)
    ious, cors = scorer_mod.main(ref_list, pred_list, scores_txt)
    return float(ious.mean()), float(cors.mean())


def merge_spans(spans: List[List[int]], gap: int = 0) -> List[List[int]]:
    if not spans:
        return []
    spans = sorted([[int(s), int(e)] for s, e in spans if e > s], key=lambda x: (x[0], x[1]))
    out = [spans[0]]
    for s, e in spans[1:]:
        ps, pe = out[-1]
        if s <= pe + gap:
            out[-1][1] = max(pe, e)
        else:
            out.append([s, e])
    return out


def predict_with_params(model_dir: str, ref_jsonl: str, out_jsonl: str):
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForTokenClassification.from_pretrained(model_dir)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    records = []
    with open(ref_jsonl, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    out_rows = []
    with torch.no_grad():
        for rec in records:
            enc = tokenizer(
                rec["model_output_text"],
                return_offsets_mapping=True,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH,
            )
            offsets = enc.pop("offset_mapping")[0]
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits[0]
            probs = F.softmax(logits / max(BEST_TEMPERATURE, 1e-6), dim=-1)

            soft_labels, hard_spans = [], []
            for j, off in enumerate(offsets):
                s, e = off[0].item(), off[1].item()
                if s >= e:
                    continue
                if probs.shape[-1] > 2:
                    p_hall = float((probs[j, 1] + probs[j, 2]).item())
                elif probs.shape[-1] == 2:
                    p_hall = float(probs[j, 1].item())
                else:
                    p_hall = 0.0
                soft_labels.append({"start": s, "end": e, "prob": p_hall})
                if p_hall >= BEST_THRESHOLD:
                    hard_spans.append([s, e])

            hard_spans = [sp for sp in hard_spans if (sp[1] - sp[0]) >= BEST_MIN_SPAN_LEN]
            hard_spans = merge_spans(hard_spans, gap=BEST_MERGE_GAP)
            out_rows.append({"id": rec["id"], "soft_labels": soft_labels, "hard_labels": hard_spans})

    write_jsonl(out_jsonl, out_rows)


clone_if_missing("https://github.com/Helsinki-NLP/mu-shroom", "mu-shroom")
KIT = os.path.abspath("mu-shroom/participant_kit")
scorer = load_scorer_module(KIT)
print("Using participant kit:", KIT)

Using participant kit: /content/mu-shroom/participant_kit


In [11]:
# Run cross-lingual EN-transfer evaluation only
if not os.path.isdir(EN_MODEL_DIR):
    raise FileNotFoundError(f"Expected English model at {EN_MODEL_DIR}. Train/copy it first.")

rows = []
for lang in TARGET_LANGS:
    print(f"\n=== Language: {lang} ===")
    ds_lang = load_dataset("Helsinki-NLP/mu-shroom", lang)

    ref_jsonl = os.path.join(OUT_ROOT, f"reference_{lang}_validation.jsonl")
    write_jsonl(
        ref_jsonl,
        [
            {
                "id": row["id"],
                "model_output_text": row["model_output_text"],
                "soft_labels": row["soft_labels"],
                "hard_labels": row["hard_labels"],
            }
            for row in ds_lang["validation"]
        ],
    )

    pred_transfer = os.path.join(OUT_ROOT, f"pred_transfer_en_to_{lang}.jsonl")
    score_transfer = os.path.join(OUT_ROOT, f"scores_transfer_en_to_{lang}.txt")
    predict_with_params(EN_MODEL_DIR, ref_jsonl, pred_transfer)
    iou_t, cor_t = score_predictions(scorer, ref_jsonl, pred_transfer, score_transfer)
    rows.append({
        "Language": lang,
        "Setting": "EN-transfer",
        "IoU": iou_t,
        "Correlation": cor_t,
        "Prediction file": pred_transfer,
    })
    print(f"EN-transfer -> IoU: {iou_t:.6f}, Cor: {cor_t:.6f}")

results_df = pd.DataFrame(rows).sort_values(by=["Language"]).reset_index(drop=True)
print("\nPer-language official EN-transfer results:")
display(results_df)


=== Language: hi ===


hi/validation-00000-of-00001.parquet:   0%|          | 0.00/52.9k [00:00<?, ?B/s]

hi/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/50 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/150 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

EN-transfer -> IoU: 0.421694, Cor: 0.398120

Per-language official EN-transfer results:


,Language,Setting,IoU,Correlation,Prediction file
0,hi,EN-transfer,0.421694,0.39812,/content/mushroom_multilingual/pred_transfer_e...
